# C/2023 A3 (Tsuchinshan–ATLAS): PLUTO MHD Simulation Log

**Reference epoch:** 2024-10-10 12:54 UTC  
**PLUTO version:** 4.4-patch3; modified solver tree: `PLUTO-comet`  
**Python environment:** Python 3.12, PyPLUTO 1.2.6, `astroquery`, `astropy`

This notebook records the setup, assumptions, code changes, completed simulations, diagnostics, figure-generation workflow, and proposed next experiment. It distinguishes completed work from experiments that were only proposed.

## 1. Project paths and latest simulation

| Purpose | Absolute path | Status |
|---|---|---|
| Original, unmodified PLUTO solver | `/home/santy/src/PLUTO` | Preserved as reference |
| Modified solver | `/home/santy/src/PLUTO-comet` | Used to compile cases |
| Main project | `/home/santy/simulaciones/C2023A3_20241010_1254` | Project root |
| C0 baseline, source disabled | `/home/santy/simulaciones/C2023A3_20241010_1254/results/C0_256/` | Completed and archived |
| C1 Haser mass-loading case | `/home/santy/simulaciones/C2023A3_20241010_1254/C1_256/` | Compiled and partially run; interrupted after negative-pressure warning |
| C1 reduced-CFL test | `/home/santy/simulaciones/C2023A3_20241010_1254/C1_256_dtcheck/` | Proposed; creation/execution not confirmed |

**Latest completed simulation:** `C1_256`  
`/home/santy/simulaciones/C2023A3_20241010_1254/C1_256/`

**Latest proposed case path:** `C1_256_dtcheck`  
`/home/santy/simulaciones/C2023A3_20241010_1254/C1_256_dtcheck/`

The second path is the planned time-step sensitivity experiment, not a confirmed completed run.

## 2. Ephemeris and upstream solar-wind state

A JPL Horizons query was made for C/2023 A3 at 2024-10-10 12:54 UTC.

Heliocentric ecliptic position:
\[
\mathbf r=(0.499055328,\;0.137594360,\;0.047987181)\;\mathrm{AU}
\]

Heliocentric velocity:
\[
\mathbf v=(35.17073698,\;-30.22329042,\;35.53058265)\;\mathrm{km\,s^{-1}}
\]

Heliocentric distance: \(r_h=0.519895372\) AU. Orbital speed: 58.41960567 km/s.

The solar-wind radial speed was assumed to be 400 km/s. In the comet-relative local frame, the flow is:
\[
\mathbf V_{rel}=(374.405605,\;0,\;0)\;\mathrm{km\,s^{-1}}
\]

A Parker-spiral field was estimated assuming a 7 nT field at 1 AU and a solar rotation period of 25.38 days. In the local simulation frame:
\[
\mathbf B=(16.49881045,\;-11.55839307,\;1.56963317)\;\mathrm{nT}
\]
with magnitude about 20.2057 nT. The assumed solar-wind density at the comet was 18.49856762 cm^-3.

**Caveat:** these are model-derived upstream conditions, not in-situ measurements at the comet for that exact time.

## 3. Code-unit normalization and initialized state

| Scale | Value |
|---|---:|
| Length, L0 | 1.0 × 10^11 cm = 1.0 × 10^6 km |
| Velocity, V0 | 4.0 × 10^7 cm/s = 400 km/s |
| Time, T0 | 2.5 × 10^3 s |
| Density, rho0 | 3.09411097591 × 10^-23 g/cm³ |
| Magnetic field, B0 | 78.87381846 nT |

Dimensionless upstream state:
- density = 1.0
- velocity = (0.936014013, 0, 0)
- pressure = 0.01032
- magnetic field = (0.20917981, -0.14654284, 0.01990056)
- initial tracer = 0

The pressure used the adopted equal proton/electron temperature assumption of 10^5 K.

## 4. Neutral coma and mass-loading model

The neutral density was represented with a Haser-type profile:
\[
n_n(r)=\frac{Q}{4\pi V_n r_{eff}^{2}}\exp(-r/L_i),
\qquad r_{eff}=\sqrt{r^2+r_{soft}^2}
\]

Adopted parameters:
- production rate: \(Q=10^{29}\) molecules/s (assumption);
- neutral speed: \(V_n=0.7\) km/s;
- ionization frequency: \(\nu_i=3.06\times10^{-6}\) s^-1;
- ionization length: \(L_i=V_n/\nu_i\approx2.2876\times10^{10}\) cm (about 228,758 km);
- softening radius: \(r_{soft}=1.2\times10^9\) cm (12,000 km);
- representative ion mass: \(m_i=18m_p\), representing H2O+.

Mass injection is \(S_\rho=m_i\nu_i n_n\). Conservative source terms add mass, momentum \(S_\rho\mathbf u_n\), and energy \(S_\rho(\epsilon_n+|\mathbf u_n|^2/2)\). The tracer source is set equal to the mass source.

**Limitations:** this is a 2D planar slice of a 3D coma, invariant in the omitted direction. Its 2D integral is not the global 3D production rate. Detailed pickup-ion heating, electron/ion thermodynamics, and additional chemistry are not represented in this simplified prescription.

## 5. Solver configuration and code modifications

Reported configuration:
- 2D Cartesian MHD; ideal-gas EOS;
- RK2 time stepping;
- flat reconstruction; MC limiter;
- GLM divergence cleaning (`DIV_CLEANING`);
- one passive tracer;
- HLL Riemann solver;
- 256 × 256 grid;
- x1 domain [-0.5, 2.5], x2 domain [-1.0, 1.0];
- x1 lower boundary: user-defined solar-wind inflow; x1 upper: outflow;
- x2 boundaries: outflow; x3: periodic with one cell.

The case Makefile points to `/home/santy/src/PLUTO-comet`. The original `/home/santy/src/PLUTO` tree was kept unchanged. A `local_make` file was used to include the GLM object:
```make
VPATH += $(SRC)/MHD/GLM
OBJ += glm.o
```

`init.c` initializes uniform solar-wind density, velocity, pressure, magnetic field, zero GLM scalar, and zero tracer. `UserDefBoundary()` applies the same upstream state at the left x1 boundary.

In `PLUTO-comet/Src/MHD/rhs_source.c`, the comet source block is guarded for Cartesian MHD and applied only during the IDIR sweep. It calls `EvaluateCometSource(x1[i], x2[j])` and adds:
- `dt * srho` to density;
- `dt * srho * unx` and `dt * srho * uny` to x1/x2 momentum;
- `dt * energy` to total energy when enabled;
- `dt * srho` to the tracer when a tracer exists.

The inspected block was around lines 255–286.

## 6. Simulation variants and execution history

### C0 — baseline without comet mass loading
The source was disabled using `COMET_QM0 = 0.0` in the then-current case configuration. C0 completed and was archived in:
`/home/santy/simulaciones/C2023A3_20241010_1254/results/C0_256/`

C0 output inspection showed uniform values: rho=1, vx1=0.936014013, vx2=vx3=0, pressure=0.01032, GLM scalar=0, tracer=0. This validated the uniform upstream initialization.

### C1 — Haser mass-loading source enabled
C1 compiled successfully in:
`/home/santy/simulaciones/C2023A3_20241010_1254/C1_256/`

It was run but manually interrupted after a negative-pressure warning at step 4975, RK stage 2, at code time about 11.807. The warning location was reported as [i,j]=[70,257], [x1,x2]=[0.302734,0.996094], with a very large Mach number. The requested `tstop` was 16. The last `dbl.out` record shown was at code time 10.30872; this differs from the warning time because output records are not written every time step.

## 7. C1 output statistics and pressure-floor analysis

For C1 output 1, PyPLUTO reported:

| Field | Minimum | Maximum | Mean |
|---|---:|---:|---:|
| rho | 0.565457876 | 76.9662784 | 2.20120492 |
| vx1 | 0.08640175 | 1.01839217 | 0.615272356 |
| vx2 | -0.427618087 | 0.335288718 | 0.0335733958 |
| vx3 | -0.0193102349 | 0.0164506661 | -0.00220185733 |
| pressure | 1.0e-12 | 0.675943095 | 0.07425688 |
| GLM scalar | -0.0771448381 | 0.0976599940 | 0.00928959249 |
| tracer | 6.63614186e-6 | 0.997060265 | 0.253901364 |

There were 2,800 pressure-floor cells out of 65,536:
\[
100\frac{2800}{65536}=4.2724609375\%
\]

The pressure floor is `g_smallPressure = 1.e-12` in `Src/globals.h`. In the MHD conservative-to-primitive mapper, negative pressure computed from total energy is replaced by this floor, and the energy is recomputed. Therefore, floor cells flag a problematic conservative energy state; the floor is not a physical pressure result.

The pressure-floor cells occurred in both lower-left and upper-right regions:
- lower-left: 1,087 cells;
- upper-right: 1,713 cells.

## 8. Energy and source-term diagnostics

The whole-domain means from `diagnose_energy.py` were:
- density: 2.201205;
- pressure: 0.07425688;
- kinetic-energy diagnostic: 0.3756991;
- magnetic-energy diagnostic: 0.1202099;
- magnetic-field magnitude: 0.4676776;
- plasma beta: 0.7636118;
- tracer: 0.2539014.

The lower-left pressure-floor region had low mean tracer (about 0.00328), while the upper-right region had much higher mean tracer (about 0.28738). Thus pressure-floor cells occur both where cometary tracer is scarce and where it is abundant. The issue cannot be attributed only to strong local mass injection.

`diagnose_source_cells.py` reported:

| Region | Mean source Sρ | Mean Sρ/rho | Mean dt·Sρ/rho |
|---|---:|---:|---:|
| Lower-left floor cells | 0.0036342 | 0.0037925 | 8.4392e-6 |
| Upper-right floor cells | 3.0689e-6 | 3.0451e-6 | 6.7760e-9 |
| All floor cells | 0.0014127 | 0.0014742 | 3.2804e-6 |

Across the whole domain, maximum source was about 509.11 at the nucleus and maximum Sρ/rho about 461.94. Using the last reported C1 time step 2.225228e-3 gives an estimated maximum fractional source increment of about 1.028 at the most strongly forced cell.

**Interpretation:** the source is stiff close to the nucleus, but pressure-floor cells have small local source increments. This makes a time-step sensitivity test worthwhile, but does not establish that source stiffness is the sole cause.

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import pyPLUTO as pp

CASE_DIR = Path("/home/santy/simulaciones/C2023A3_20241010_1254/C1_256")
OUTPUT_NUMBER = 1
D = pp.Load(path=str(CASE_DIR) + "/", nout=OUTPUT_NUMBER)
print("Loaded:", CASE_DIR, "output:", OUTPUT_NUMBER)

def get_field(data, *names):
    for name in names:
        if hasattr(data, name):
            return np.asarray(getattr(data, name))
    raise AttributeError(f"Could not find any of: {names}. Inspect dir(D).")

rho = get_field(D, "rho", "Rho")
vx1 = get_field(D, "vx1", "Vx1")
vx2 = get_field(D, "vx2", "Vx2")
prs = get_field(D, "prs", "P")
tr1 = get_field(D, "tr1", "Tr1")

x1 = np.asarray(D.x1) if hasattr(D, "x1") else np.arange(rho.shape[1])
x2 = np.asarray(D.x2) if hasattr(D, "x2") else np.arange(rho.shape[0])
print("Field shape:", rho.shape)


In [ ]:
def plot_map(field, title, cbar_label, filename, log_scale=False,
             cmap="viridis", vmin=None, vmax=None):
    a = np.asarray(field)
    shown = np.log10(np.maximum(a, 1e-30)) if log_scale else a
    fig, ax = plt.subplots(figsize=(9, 5.5))
    m = ax.pcolormesh(x1, x2, shown, shading="auto", cmap=cmap,
                      vmin=vmin, vmax=vmax)
    ax.set_xlabel("x1 (code units)")
    ax.set_ylabel("x2 (code units)")
    ax.set_title(title)
    cb = fig.colorbar(m, ax=ax)
    cb.set_label(("log10 " if log_scale else "") + cbar_label)
    fig.tight_layout()
    fig.savefig(filename, dpi=200, bbox_inches="tight")
    plt.show()

plot_map(rho, "C1: mass density", "rho", "C1_density.png", log_scale=True)
plot_map(prs, "C1: thermal pressure", "pressure", "C1_pressure.png", log_scale=True)
plot_map(tr1, "C1: cometary tracer", "tr1", "C1_tracer.png", vmin=0, vmax=1)

speed = np.sqrt(vx1**2 + vx2**2)
plot_map(speed, "C1: in-plane speed", "|v|", "C1_speed.png")

floor = prs <= 1.000001e-12
plot_map(floor.astype(float), "C1: pressure-floor mask",
         "floor-cell flag", "C1_pressure_floor_mask.png",
         cmap="gray_r", vmin=0, vmax=1)


In [ ]:
def report_stats(name, a):
    a = np.asarray(a)
    finite = a[np.isfinite(a)]
    print(f"{name:12s} min={finite.min():.8e} max={finite.max():.8e} "
          f"mean={finite.mean():.8e} NaN={np.isnan(a).sum()}")

for name, a in [("rho", rho), ("vx1", vx1), ("vx2", vx2),
                ("pressure", prs), ("tracer", tr1), ("speed", speed)]:
    report_stats(name, a)

print(f"Pressure-floor cells: {floor.sum()}")
print(f"Affected percentage: {100*floor.mean():.8f}%")


## 9. Generate plots from PLUTO outputs

Run these cells in the `pypluto` virtual environment. They load C1 output 1 and generate density, pressure, tracer, speed, and pressure-floor-mask images. If PyPLUTO exposes different field names in another output build, inspect `dir(D)` and adjust the field names.

## 10. Proposed time-step sensitivity test

A separate case was proposed at:
`/home/santy/simulaciones/C2023A3_20241010_1254/C1_256_dtcheck/`

Proposed changes only:
- CFL: 0.3 → 0.03;
- `first_dt`: 1e-4 → 1e-5;
- `tstop`: 16.0 → 4.0 for a shorter diagnostic run;
- `dbl` output interval: 10.31 → 0.5.

All physical parameters, grid, source model, and boundary conditions should remain unchanged. The experiment was proposed but no successful compile/run output was supplied, so it must remain labelled **planned / unconfirmed**.

If reducing the time step substantially reduces the pressure-floor count or delays the negative-pressure warning, temporal resolution is implicated. If the issue persists in similar locations, investigate conservative energy updates, boundary effects, magnetic-energy evolution, and source-term integration independently. Raising the pressure floor would mask the issue rather than diagnose it.

## 11. Reproducibility checklist

- [ ] Confirm the case compiles against `/home/santy/src/PLUTO-comet`.
- [ ] Preserve `/home/santy/src/PLUTO` as the original solver.
- [ ] Use a separate directory for every experiment.
- [ ] Archive `definitions.h`, `pluto.ini`, `init.c`, `comet_parameters.h`, `comet_source.h`, Makefile, and `local_make` for each case.
- [ ] Preserve `dbl.out`, `grid.out`, `data.*.dbl`, `restart.out`, and terminal logs.
- [ ] Record whether each run reached `tstop`.
- [ ] Re-run the plotting and diagnostic cells for each output.
- [ ] Clearly label assumptions versus measured/observational constraints.

**Project status at the time of this notebook:** C0 baseline validated; C1 compiled and partially evolved; C1 showed pressure-floor cells and a negative-pressure warning; a reduced-CFL sensitivity test was proposed but not confirmed as executed.